# Lab 14 — Event-Time Windows and Watermarks

**Tujuan:** Membedakan event time, windowing, watermark, dan state.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 14")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Built-in timestamped stream

Demo ini memperlihatkan window dan watermark dalam execution plan; karena rate source menghasilkan event tepat waktu, **demo ini tidak membuktikan pembuangan late events**.

In [ ]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
with_group = stream.withColumn("group_id", F.col("value") % 3)
windows = with_group.withWatermark("timestamp", "10 seconds")
summary = windows.groupBy(
    F.window("timestamp", "10 seconds"),
    "group_id"
).count()
summary.explain("formatted")

## 2. Observe completed/ongoing windows

Gunakan complete mode untuk demonstrasi agregasi. Memory sink tidak fault tolerant dan cocok hanya untuk lab.

In [ ]:
import time
query = (
    summary.writeStream
    .format("memory")
    .queryName("lab14_windows")
    .outputMode("complete")
    .start()
)
time.sleep(15)
spark.sql("""
    SELECT window.start AS start_time, group_id, count
    FROM lab14_windows
    ORDER BY start_time, group_id
""").show(20, truncate=False)
print("Watermark:", (query.lastProgress or {}).get("eventTime"))
query.stop()

## 3. Discussion

Watermark membatasi seberapa lama Spark perlu mempertahankan state berdasarkan event-time progress. Dalam produksi, test late events membutuhkan data dengan timestamp out-of-order, bukan sekadar `rate` source.

**Tugas:** jelaskan apa yang terjadi jika window diperbesar dan mengapa stateful aggregation lebih kompleks daripada streaming `select`/`filter`.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.